In [1]:
from pathlib import Path
import platform
from typing import List
import tifffile as tiff
from typing import List, Optional

"C:\Users\cowboy\OneDrive\Documents\Unviversity of Alabama\Nuclear_Scaling\Data_Sets\Control\Extract_3\1-5.tif"

In [2]:


SYSTEM = platform.system()

if SYSTEM == "Linux":
    DATA_ROOT = Path("/home/tdeibert/Data")
else:
    DATA_ROOT = Path(r"C:/Users/cowboy/OneDrive/Documents/Unviversity of Alabama/")

INPUT_DIR = DATA_ROOT / "Nuclear_Scaling/Data_Sets/Control/Extract_3"
OUTPUT_DIR = DATA_ROOT / "Nuclear_Scaling/Data_Sets/Control/Extract_3"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Manual time order
time_block_1 = INPUT_DIR / "1-5.tif"
time_block_2 = INPUT_DIR / "6-10.tif"


input_files = [
    time_block_2,
    time_block_1,
]

for f in input_files:
    if not f.exists():
        raise FileNotFoundError(f"Missing file: {f}")

output_file = OUTPUT_DIR / "control_extract_1.1.tif"


In [3]:
import tifffile as tiff

arr = tiff.memmap(time_block_1)
print(arr.shape)

(5, 20, 3, 3889, 5732)


In [ ]:

from weakref import ref

from matplotlib.pylab import ndim


def concatenate_tiff_series(
    input_paths: List[Path],
    output_path: Path,
    chunk_size: int = 1,
    concat_axis: Optional[int] = None,
    concat_dim: Optional[str] = "T",
) -> Path:
    """
    Concatenate TIFF series along one axis using memory mapping.

    Parameters
    ----------
    input_paths : list[Path]
        Ordered list of input TIFF files in the order they should be appended.
    output_path : Path
        Output TIFF path.
    chunk_size : int
        Number of slices/frames to copy at once along the concat axis.
    concat_axis : int, optional
        Integer axis to concatenate along. Overrides concat_dim if provided.
    concat_dim : str, optional
        Axis label to concatenate along, e.g. "T".
        Only used if TIFF metadata exposes axis labels.

    Returns
    -------
    Path
        Path to saved output TIFF.
    """
    if not input_paths:
        raise ValueError("No input files provided.")

    input_paths = [Path(p) for p in input_paths]
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    arrays = []
    series_shapes = []
    series_axes = []

    for p in input_paths:
        if not p.exists():
            raise FileNotFoundError(f"Missing file: {p}")

        with tiff.TiffFile(p) as tif:
            series = tif.series[0]
            series_shapes.append(series.shape)
            series_axes.append(getattr(series, "axes", None))

        arr = tiff.memmap(p)
        arrays.append(arr)

    ref = arrays[0]
    ref_shape = ref.shape
    ref_dtype = ref.dtype
    ndim = ref.ndim
    ref_axes = series_axes[0]

    if ndim != 5:
        raise ValueError(
            f"ImageJ hyperstack metadata block expects 5D data, but got shape {ref_shape}"
        )

    ref_t, ref_z, ref_c, ref_y, ref_x = ref_shape

    print("Input files:")
    for p, arr, shp, ax in zip(input_paths, arrays, series_shapes, series_axes):
        print(f"  {p.name}: memmap_shape={arr.shape}, series_shape={shp}, axes={ax}, dtype={arr.dtype}")

    # Resolve concat axis
    if concat_axis is None:
        if concat_dim is not None and ref_axes is not None:
            if concat_dim in ref_axes:
                concat_axis = ref_axes.index(concat_dim)
            else:
                raise ValueError(
                    f"Requested concat_dim='{concat_dim}', but first file axes are {ref_axes}"
                )
        else:
            raise ValueError(
                "Could not determine concat axis automatically. "
                "Please provide concat_axis explicitly."
            )

    if concat_axis < 0:
        concat_axis += ndim

    if concat_axis < 0 or concat_axis >= ndim:
        raise ValueError(f"concat_axis={concat_axis} is out of bounds for ndim={ndim}")

    # Validate all inputs
    for p, arr, ax in zip(input_paths[1:], arrays[1:], series_axes[1:]):
        if arr.ndim != ndim:
            raise ValueError(
                f"{p.name} ndim mismatch: got {arr.ndim}, expected {ndim}"
            )

        if arr.dtype != ref_dtype:
            raise ValueError(
                f"{p.name} dtype mismatch: got {arr.dtype}, expected {ref_dtype}"
            )

        if ref_axes is not None and ax is not None and ax != ref_axes:
            raise ValueError(
                f"{p.name} axes mismatch: got {ax}, expected {ref_axes}"
            )

        for d in range(ndim):
            if d == concat_axis:
                continue
            if arr.shape[d] != ref_shape[d]:
                raise ValueError(
                    f"{p.name} shape mismatch at axis {d}: "
                    f"got {arr.shape[d]}, expected {ref_shape[d]}"
                )

    # Build output shape
    out_shape = list(ref_shape)
    out_shape[concat_axis] = sum(arr.shape[concat_axis] for arr in arrays)
    out_shape = tuple(out_shape)

    print(f"\nUsing concat_axis={concat_axis}")
    print(f"Reference shape: {ref_shape}")
    print(f"Output shape:    {out_shape}")
    print(f"Reference axes:  {ref_axes}")

    # Preallocate output on disk with ImageJ hyperstack metadata
    out = tiff.memmap(
    output_path,
    shape=out_shape,
    dtype=ref_dtype,
    bigtiff=True,
    imagej=True,
    metadata={
        "axes": "TZCYX",
        "channels": ref_c,
        "slices": ref_z,
        "frames": total_t,
        "hyperstack": True,
        "mode": "grayscale",
    },
)

    # Copy data chunk-by-chunk
    out_start = 0
    for p, arr in zip(input_paths, arrays):
        axis_len = arr.shape[concat_axis]
        print(f"Copying {p.name} into output axis range [{out_start}:{out_start + axis_len}]")

        for local_start in range(0, axis_len, chunk_size):
            local_stop = min(local_start + chunk_size, axis_len)
            n = local_stop - local_start
            out_stop = out_start + n

            src_slices = [slice(None)] * ndim
            dst_slices = [slice(None)] * ndim

            src_slices[concat_axis] = slice(local_start, local_stop)
            dst_slices[concat_axis] = slice(out_start, out_stop)

            out[tuple(dst_slices)] = arr[tuple(src_slices)]
            out_start = out_stop

    out.flush()
    print(f"\nSaved combined TIFF to:\n{output_path}")
    return output_path

In [4]:
from pathlib import Path
from typing import List
import tifffile as tiff


def concatenate_hyperstacks_in_time(
    input_paths: List[Path],
    output_path: Path,
    chunk_t: int = 1,
) -> Path:
    """
    Concatenate 5D TIFF hyperstacks along time axis.

    Expected input axis order:
        (T, Z, C, Y, X)

    Output is written as an ImageJ/Fiji-compatible hyperstack.

    Parameters
    ----------
    input_paths : list[Path]
        Ordered list of input TIFF files in the exact order they should appear
        in the output.
    output_path : Path
        Output TIFF path.
    chunk_t : int
        Number of timepoints to copy at once.

    Returns
    -------
    Path
        Path to saved output TIFF.
    """
    if not input_paths:
        raise ValueError("No input files provided.")

    input_paths = [Path(p) for p in input_paths]
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    arrays = []
    metadata_list = []

    for p in input_paths:
        if not p.exists():
            raise FileNotFoundError(f"Missing file: {p}")

        with tiff.TiffFile(p) as tif:
            series = tif.series[0]
            axes = getattr(series, "axes", None)
            shape = series.shape
            dtype = series.dtype

        arr = tiff.memmap(p)
        arrays.append(arr)
        metadata_list.append((shape, axes, dtype))

    ref = arrays[0]
    ref_shape = ref.shape
    ref_dtype = ref.dtype
    ref_ndim = ref.ndim
    ref_axes = metadata_list[0][1]

    if ref_ndim != 5:
        raise ValueError(
            f"Expected 5D input shaped like (T, Z, C, Y, X), but got {ref_shape}"
        )

    if ref_axes != "TZCYX":
        raise ValueError(
            f"Expected axes='TZCYX', but got {ref_axes}"
        )

    ref_t, ref_z, ref_c, ref_y, ref_x = ref_shape

    print("Input files:")
    for p, arr, meta in zip(input_paths, arrays, metadata_list):
        shape, axes, dtype = meta
        print(
            f"  {p.name}: memmap_shape={arr.shape}, "
            f"series_shape={shape}, axes={axes}, dtype={dtype}"
        )

    for p, arr, meta in zip(input_paths[1:], arrays[1:], metadata_list[1:]):
        shape, axes, dtype = meta

        if arr.ndim != 5:
            raise ValueError(f"{p.name} is not 5D. Got shape {arr.shape}")

        if axes != "TZCYX":
            raise ValueError(f"{p.name} axes mismatch: expected TZCYX, got {axes}")

        if arr.dtype != ref_dtype:
            raise ValueError(
                f"{p.name} dtype mismatch: got {arr.dtype}, expected {ref_dtype}"
            )

        _, z, c, y, x = arr.shape
        if (z, c, y, x) != (ref_z, ref_c, ref_y, ref_x):
            raise ValueError(
                f"{p.name} has incompatible shape {arr.shape}. "
                f"Expected (*, {ref_z}, {ref_c}, {ref_y}, {ref_x})"
            )

    total_t = sum(arr.shape[0] for arr in arrays)
    out_shape = (total_t, ref_z, ref_c, ref_y, ref_x)

    print(f"\nOutput shape: {out_shape}")
    print(f"Writing Fiji/ImageJ hyperstack metadata: axes=TZCYX, T={total_t}, Z={ref_z}, C={ref_c}")

    # Create output memmap with ImageJ metadata
    out = tiff.memmap(
        output_path,
        shape=out_shape,
        dtype=ref_dtype,
        bigtiff=True,
        imagej=True,
        metadata={
            "axes": "TZCYX",
            "channels": ref_c,
            "slices": ref_z,
            "frames": total_t,
            "hyperstack": True,
            "mode": "grayscale",
        },
    )

    out_t_start = 0
    for p, arr in zip(input_paths, arrays):
        t_size = arr.shape[0]
        print(f"Copying {p.name} into output T[{out_t_start}:{out_t_start + t_size}]")

        for local_t_start in range(0, t_size, chunk_t):
            local_t_stop = min(local_t_start + chunk_t, t_size)
            n_t = local_t_stop - local_t_start
            out_t_stop = out_t_start + n_t

            out[out_t_start:out_t_stop, :, :, :, :] = arr[local_t_start:local_t_stop, :, :, :, :]
            out_t_start = out_t_stop

    out.flush()
    print(f"\nSaved combined hyperstack to:\n{output_path}")
    return output_path

In [5]:
input_files = [
    INPUT_DIR / "6-10.tif",
    INPUT_DIR / "1-5.tif",
]

output_file = OUTPUT_DIR / "control_extract_1.1.tif"

print("\nPlanned concatenation order:")
for i, f in enumerate(input_files, start=1):
    print(f"{i}: {f.name}")

concatenate_hyperstacks_in_time(
    input_paths=input_files,
    output_path=output_file,
    chunk_t=1
)


Planned concatenation order:
1: 6-10.tif
2: 1-5.tif
Input files:
  6-10.tif: memmap_shape=(5, 20, 3, 3889, 5732), series_shape=(5, 20, 3, 3889, 5732), axes=TZCYX, dtype=uint16
  1-5.tif: memmap_shape=(5, 20, 3, 3889, 5732), series_shape=(5, 20, 3, 3889, 5732), axes=TZCYX, dtype=uint16

Output shape: (10, 20, 3, 3889, 5732)
Writing Fiji/ImageJ hyperstack metadata: axes=TZCYX, T=10, Z=20, C=3


c:\Users\cowboy\anaconda3\envs\ml_env_tf_2.15\lib\site-packages\tifffile\tifffile.py:1753: UserWarning: <tifffile.TiffWriter 'control_extract_1.1.tif'> writing nonconformant BigTIFF ImageJ
  warnings.warn(


Copying 6-10.tif into output T[0:5]
Copying 1-5.tif into output T[5:10]

Saved combined hyperstack to:
C:\Users\cowboy\OneDrive\Documents\Unviversity of Alabama\Nuclear_Scaling\Data_Sets\Control\Extract_3\control_extract_1.1.tif


WindowsPath('C:/Users/cowboy/OneDrive/Documents/Unviversity of Alabama/Nuclear_Scaling/Data_Sets/Control/Extract_3/control_extract_1.1.tif')

In [ ]:
concatenate_tiff_series(
    input_paths=input_files,
    output_path=output_file,
    chunk_size=1,
    concat_dim="T"
)

In [6]:
import napari
import tifffile as tiff

combined_img = tiff.memmap(output_file)

viewer = napari.Viewer()

viewer.add_image(
    combined_img,
    name=["Channel_1", "Channel_2", "Channel_3"],
    channel_axis=2,
)

napari.run()

Split Hyperstacks Along Time 

In [ ]:
def split_hyperstack_by_time(img, block_size):
    """
    Split a TZCYX array into sequential time blocks.
    """
    blocks = []
    n_t = img.shape[0]

    for start in range(0, n_t, block_size):
        stop = min(start + block_size, n_t)
        blocks.append(img[start:stop, :, :, :, :])

    return blocks

Single Channel Extraction 

In [ ]:
def extract_channel(img, channel_idx):
    """
    Extract one channel from a TZCYX hyperstack.
    Returns shape: (T, Z, Y, X)
    """
    return img[:, :, channel_idx, :, :]

Extract Single Time Point 

In [ ]:
def extract_timepoint(img, t_idx):
    """
    Extract one timepoint from a TZCYX hyperstack.
    Returns shape: (Z, C, Y, X)
    """
    return img[t_idx, :, :, :, :]

Extract a Single Z plane across time 

In [ ]:
def extract_z_plane(img, z_idx):
    """
    Extract one z-plane across all timepoints.
    Returns shape: (T, C, Y, X)
    """
    return img[:, z_idx, :, :, :]

Deinterleave Time Points

In [ ]:
def deinterleave_time(img, n_streams):
    """
    Split a TZCYX hyperstack into interleaved time streams.

    Example:
        n_streams=2 splits timepoints into [0,2,4,...] and [1,3,5,...]
    """
    return [img[i::n_streams, :, :, :, :] for i in range(n_streams)]

Deinterleave channels 

In [ ]:
def deinterleave_channels(img, n_streams):
    """
    Split channel axis into interleaved groups.
    """
    return [img[:, :, i::n_streams, :, :] for i in range(n_streams)]

Batch Processing

In [ ]:
from pathlib import Path
from typing import List
import pandas as pd
import tifffile as tiff


def concatenate_hyperstacks_in_time(
    input_paths: List[Path],
    output_path: Path,
    chunk_t: int = 1,
) -> Path:
    """
    Concatenate 5D TIFF hyperstacks along time axis.
    Expected axes: TZCYX
    """
    if not input_paths:
        raise ValueError("No input files provided.")

    input_paths = [Path(p) for p in input_paths]
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    arrays = []
    metadata_list = []

    for p in input_paths:
        if not p.exists():
            raise FileNotFoundError(f"Missing file: {p}")

        with tiff.TiffFile(p) as tif:
            series = tif.series[0]
            axes = getattr(series, "axes", None)
            shape = series.shape
            dtype = series.dtype

        arr = tiff.memmap(p)
        arrays.append(arr)
        metadata_list.append((shape, axes, dtype))

    ref = arrays[0]
    ref_shape = ref.shape
    ref_dtype = ref.dtype
    ref_axes = metadata_list[0][1]

    if ref.ndim != 5:
        raise ValueError(f"Expected 5D TZCYX input, got {ref_shape}")

    if ref_axes != "TZCYX":
        raise ValueError(f"Expected axes='TZCYX', got {ref_axes}")

    ref_t, ref_z, ref_c, ref_y, ref_x = ref_shape

    for p, arr, meta in zip(input_paths[1:], arrays[1:], metadata_list[1:]):
        shape, axes, dtype = meta

        if arr.ndim != 5:
            raise ValueError(f"{p.name} is not 5D. Got shape {arr.shape}")
        if axes != "TZCYX":
            raise ValueError(f"{p.name} axes mismatch: expected TZCYX, got {axes}")
        if arr.dtype != ref_dtype:
            raise ValueError(f"{p.name} dtype mismatch: {arr.dtype} != {ref_dtype}")

        _, z, c, y, x = arr.shape
        if (z, c, y, x) != (ref_z, ref_c, ref_y, ref_x):
            raise ValueError(
                f"{p.name} shape mismatch. "
                f"Expected (*, {ref_z}, {ref_c}, {ref_y}, {ref_x}), got {arr.shape}"
            )

    total_t = sum(arr.shape[0] for arr in arrays)
    out_shape = (total_t, ref_z, ref_c, ref_y, ref_x)

    print(f"\nSaving: {output_path.name}")
    print(f"Output shape: {out_shape}")
    print("Concatenation order:")
    for i, p in enumerate(input_paths, start=1):
        print(f"  {i}: {p.name}")

    out = tiff.memmap(
        output_path,
        shape=out_shape,
        dtype=ref_dtype,
        bigtiff=True,
        imagej=True,
        metadata={
            "axes": "TZCYX",
            "channels": ref_c,
            "slices": ref_z,
            "frames": total_t,
            "hyperstack": True,
            "mode": "grayscale",
        },
    )

    out_t_start = 0
    for p, arr in zip(input_paths, arrays):
        t_size = arr.shape[0]

        for local_t_start in range(0, t_size, chunk_t):
            local_t_stop = min(local_t_start + chunk_t, t_size)
            n_t = local_t_stop - local_t_start
            out_t_stop = out_t_start + n_t

            out[out_t_start:out_t_stop, :, :, :, :] = arr[local_t_start:local_t_stop, :, :, :, :]
            out_t_start = out_t_stop

    out.flush()
    return output_path


def run_batch_concat_from_manifest(
    manifest_path: Path,
    raw_dir: Path,
    output_dir: Path,
    chunk_t: int = 1,
) -> pd.DataFrame:
    """
    Batch process hyperstack concatenation jobs from a manifest CSV.

    Manifest columns:
        experiment_id, order, input_file, output_file
    """
    manifest_path = Path(manifest_path)
    raw_dir = Path(raw_dir)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    df = pd.read_csv(manifest_path)

    required_cols = {"experiment_id", "order", "input_file", "output_file"}
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"Manifest missing required columns: {missing}")

    df = df.sort_values(["experiment_id", "order"]).reset_index(drop=True)

    log_rows = []

    for experiment_id, group in df.groupby("experiment_id", sort=False):
        input_paths = [raw_dir / f for f in group["input_file"].tolist()]
        output_path = output_dir / group["output_file"].iloc[0]

        try:
            concatenate_hyperstacks_in_time(
                input_paths=input_paths,
                output_path=output_path,
                chunk_t=chunk_t,
            )
            status = "success"
            error_message = ""
        except Exception as e:
            status = "failed"
            error_message = str(e)

        log_rows.append({
            "experiment_id": experiment_id,
            "output_file": str(output_path),
            "n_inputs": len(input_paths),
            "status": status,
            "error": error_message,
        })

    log_df = pd.DataFrame(log_rows)
    log_path = output_dir / "batch_concat_log.csv"
    log_df.to_csv(log_path, index=False)

    print(f"\nBatch log saved to: {log_path}")
    return log_df

In [ ]:
from pathlib import Path
import platform

SYSTEM = platform.system()

if SYSTEM == "Linux":
    PROJECT_ROOT = Path("/home/tdeibert/Data/microscopy_batch")
else:
    PROJECT_ROOT = Path(r"C:/Users/cowboy/OneDrive/Documents/Unviversity of Alabama/microscopy_batch")

RAW_DIR = PROJECT_ROOT / "raw_data"
MANIFEST_PATH = PROJECT_ROOT / "manifests" / "concat_manifest.csv"
OUTPUT_DIR = PROJECT_ROOT / "processed"

log_df = run_batch_concat_from_manifest(
    manifest_path=MANIFEST_PATH,
    raw_dir=RAW_DIR,
    output_dir=OUTPUT_DIR,
    chunk_t=1,
)

print(log_df)